In [35]:
!pip install -U crewai

In [36]:
!pip install "crewai[google-genai]"

In [37]:
!pip install "crewai[tools]"

In [38]:
from crewai_tools import SerperDevTool

In [39]:
from google.colab import userdata
serper_tool=SerperDevTool(api_key=userdata.get("SERPER_API_KEY"))

#**Creative Game Designer Agent**
```
   role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
   
   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
```

In [40]:
from crewai import Agent, LLM, Task, Crew, Process
from google.colab import userdata

In [41]:
llm=LLM(
    model="gemini/gemini-2.5-flash",
    api_key=userdata.get("Gemini_API_KEY")
)

In [42]:
game_designer=Agent(
    role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",

   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
    llm=llm,
    verbose=True,
    tools=[serper_tool]
)

#**Senior Python Game Developer Agent**

```
role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
```

In [43]:
developer=Agent(
    role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
    llm=llm,
    verbose=True,
    tools=[serper_tool]
)

#**QA Engineer & Code Reviewer Agent**

    role="QA Engineer & Code Reviewer",

    goal="Test, review, and improve the code for bugs, playability, and completeness",
    
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",


In [44]:

QA_engineer=Agent(role="QA Engineer & Code Reviewer",

goal="Test, review, and improve the code for bugs, playability, and completeness",

backstory=
    "You are a meticulous QA engineer and code reviewer."
    "You carefully check:"
    "- Does the code run without errors?"
    "- Does it implement ALL the designed features?"
    "- Is it fun/playable? Any obvious balance issues?"
    "- Code style, variable names, comments"
    "Suggest fixes or small improvements and output the FINAL improved code.",
    llm=llm,
    verbose=True)

### **Game Designing Task**

    description=
        "Take the user's game idea: {game_idea}"
        "1. Clarify and expand it into a fun, simple 2D game"
        "2. Describe: objective, controls, entities, win/lose"
        "3. Keep scope small (one level, basic mechanics)"
        "Output format:"
        "## Game Design Document"
        "- Title: ..."
        "- Genre: ..."
        "- Objective: ..."
        "- Controls: ..."
        "- Entities: ..."
        "- Mechanics: ...",

In [45]:
task_design = Task(
    description=
    "Take the user's game idea: {game_idea}"
    "1. Clarify and expand it into a fun, simple 2D game"
    "2. Describe: objective, controls, entities, win/lose"
    "3. Keep scope small (one level, basic mechanics)"
    "Output format:"
    "## Game Design Document"
    "- Title: ..."
    "- Genre: ..."
    "- Objective: ..."
    "- Controls: ..."
    "- Entities: ..."
    "- Mechanics: ...",
    expected_output="A clear markdown Game design Document",
    agent=game_designer
)

### **Coding Task**

    description=
        "Using the game design from the previous task"
        "Write a COMPLETE, standalone Python script using Pygame that implements the game."
        "- Include import pygame, sys, random (if needed)"
        "- Full game loop, init, events, update, draw"
        "- Make it runnable with python game.py"
        "- Add simple comments"
        "- The main game loop must be exposed in the python code, it should not be inside any function like main",
        "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",

In [46]:
task_code = Task(
    description=
    "Using the game design from the previous task"
    "Write a COMPLETE, standalone Python script using Pygame that implements the game."
    "- Include import pygame, sys, random (if needed)"
    "- Full game loop, init, events, update, draw"
    "- Make it runnable with python game.py"
    "- Add simple comments"
    "- The main game loop must be exposed in the python code, it should not be inside any function like main"
    "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",
    expected_output="A complete runnable Python script",
    agent=developer,
    context=[task_design]

)

###**Review Task**

    description=
        "Review the Python code from the previous task."
        "1. Check for syntax/runtime errors"
        "2. Verify it matches the design document"
        "3. Test mentally: does it have init, loop, quit handling, drawing?"
        "4. Suggest fixes/improvements if needed"
        "5. Output the FINAL, improved, ready-to-run code"
        "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    

In [47]:
task_review = Task(
    description=
    "Review the Python code from the previous task."
    "1. Check for syntax/runtime errors"
    "2. Verify it matches the design document"
    "3. Test mentally: does it have init, loop, quit handling, drawing?"
    "4. Suggest fixes/improvements if needed"
    "5. Output the FINAL, improved, ready-to-run code"
    "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    expected_output="Final polished, runnable Pygame Python script and instructions on how to play the game",
    agent=QA_engineer,
    context=[task_design, task_code]
)

In [48]:
game_crew = Crew(
    agents = [game_designer, developer, QA_engineer],
    tasks = [task_design, task_code, task_review],
    process = Process.sequential,
    tracing=True,
    verbose=True
)

In [51]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

# NOTE: This cell currently attempts to run the CrewAI process.
# To troubleshoot the 404 error with the game server,
# we need to focus on the game deployment cells.
# This cell does not directly contribute to the 404 error in question.
# We will add a diagnostic step below instead.

# game_idea = "A fun endless runner where a character jump over obstacles"
# result = await game_crew.kickoff_async(inputs={"game_idea": game_idea})
# print(result)

print("Attempting to curl the local http.server to check its status...")
# Add a small delay to ensure the server has definitely started
import time
time.sleep(2)

# Try to fetch index.html from the local server
# -s: silent, -S: show error, -o /dev/null: discard output, -w '%{http_code}': print http code
local_response = !curl -s -S -o /dev/null -w '%{http_code}' http://localhost:8000/index.html

if local_response and local_response[0].strip() == '200':
    print("✅ Local http.server responded with HTTP 200 OK for index.html.")
    print("This indicates the server is running and serving files locally.")
    print("The problem likely lies with the ngrok tunnel or external access.")
else:
    print(f"❌ Local http.server did NOT respond with HTTP 200 OK for index.html. Response: {local_response}")
    print("This indicates the server might not be running or is not serving files correctly locally.")
    print("Please check the execution of the game deployment cell (ey6GoyYxJWIq).")

# Display the server logs again to see if the curl request was logged
print("\nUpdated HTTP Server Log after local curl attempt:")
!cat /tmp/http_server_log.txt

Attempting to curl the local http.server to check its status...
✅ Local http.server responded with HTTP 200 OK for index.html.
This indicates the server is running and serving files locally.
The problem likely lies with the ngrok tunnel or external access.

Updated HTTP Server Log after local curl attempt:
127.0.0.1 - - [13/Jun/2026 06:18:28] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 06:18:30] "GET /favicon.png HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 06:18:42] "GET /favicon.png HTTP/1.1" 304 -
127.0.0.1 - - [13/Jun/2026 06:19:01] "GET / HTTP/1.1" 304 -
127.0.0.1 - - [13/Jun/2026 06:19:02] "GET /favicon.png HTTP/1.1" 304 -
127.0.0.1 - - [13/Jun/2026 06:19:15] "GET /index.html HTTP/1.1" 200 -


In [52]:
%%writefile main.py
import pygame
import sys
import random

# --- Pygame Initialization ---
pygame.init()

# --- Constants ---
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 400
GROUND_HEIGHT = 50  # Height of the ground from the bottom of the screen
FPS = 60            # Frames per second

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
BLUE = (0, 0, 255)
GREEN = (0, 255, 0)
RED = (255, 0, 0)
GRAY = (150, 150, 150)

# Player properties
PLAYER_WIDTH = 30
PLAYER_HEIGHT = 50
PLAYER_X = 50  # Player's fixed horizontal position on the screen

# Jump properties
JUMP_STRENGTH = -12 # Negative for upward movement
GRAVITY = 0.5       # Gravity pulling the player down

# Obstacle properties
OBSTACLE_WIDTH_MIN = 20
OBSTACLE_WIDTH_MAX = 70
OBSTACLE_HEIGHT_MIN = 30
OBSTACLE_HEIGHT_MAX = 80
# These define the horizontal distance from the left edge of the screen that the
# *left edge* of the last obstacle must cross before a new one can be generated off-screen.
# This indirectly controls the density and spacing of obstacles.
MIN_DISTANCE_BETWEEN_OBSTACLES = 200
MAX_DISTANCE_BETWEEN_OBSTACLES = 400

# Game speed
INITIAL_GAME_SPEED = 5
SPEED_INCREASE_RATE = 0.001 # How much game speed increases per frame

# --- Screen Setup ---
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Pixel Dash")
clock = pygame.time.Clock()

# --- Fonts ---
font = pygame.font.Font(None, 36)         # Default font for score and instructions
game_over_font = pygame.font.Font(None, 64) # Larger font for "GAME OVER"

# --- Game Variables (initial state) ---
game_over = False
score = 0
game_speed = INITIAL_GAME_SPEED

# Player state
# Player's starting y position (on top of the ground)
player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
player_vel_y = 0       # Player's vertical velocity
is_jumping = False     # True if the player is currently in the air
# Pygame Rect for player, used for drawing and collision detection.
# For simplicity, player animation is represented by this rectangle.
player_rect = pygame.Rect(PLAYER_X, player_y, PLAYER_WIDTH, PLAYER_HEIGHT)

# List to store active obstacles. Each obstacle is a tuple: (pygame.Rect, color)
obstacles = []


# --- Helper Function to Reset Game ---
def reset_game():
    """Resets all game variables to their initial state for a new game."""
    global game_over, score, game_speed, player_y, player_vel_y, is_jumping, obstacles
    game_over = False
    score = 0
    game_speed = INITIAL_GAME_SPEED
    player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
    player_vel_y = 0
    is_jumping = False
    obstacles = []
    player_rect.y = player_y # Ensure player_rect is updated for the new game

# --- Main Game Loop ---
running = True
while running:
    # --- Event Handling ---
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.K_SPACE or event.key == pygame.K_UP:
            if game_over:
                # If game is over, pressing jump restarts the game
                reset_game()
            elif not is_jumping:
                # Only allow jump if the player is on the ground
                player_vel_y = JUMP_STRENGTH
                is_jumping = True

    # --- Game Logic (only runs if game is not over) ---
    if not game_over:
        # Player update: Apply gravity and update position
        player_vel_y += GRAVITY
        player_y += player_vel_y

        # Prevent player from falling below the ground
        if player_y >= SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT:
            player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
            player_vel_y = 0
            is_jumping = False

        # Update the player's collision rectangle
        player_rect.y = player_y

        # Obstacle generation logic
        # Generate a new obstacle if there are no obstacles, or if the last obstacle
        # has moved far enough to the left to allow for a new one with proper spacing.
        # The random.randint here defines how far to the left of SCREEN_WIDTH the
        # *left edge* of the last obstacle must be before a new one can spawn.
        if not obstacles or obstacles[-1][0].x < SCREEN_WIDTH - random.randint(MIN_DISTANCE_BETWEEN_OBSTACLES, MAX_DISTANCE_BETWEEN_OBSTACLES):
            obstacle_width = random.randint(OBSTACLE_WIDTH_MIN, OBSTACLE_WIDTH_MAX)
            obstacle_height = random.randint(OBSTACLE_HEIGHT_MIN, OBSTACLE_HEIGHT_MAX)
            # Start obstacles slightly off-screen to the right
            obstacle_x = SCREEN_WIDTH + random.randint(0, 100)
            obstacle_y = SCREEN_HEIGHT - GROUND_HEIGHT - obstacle_height
            new_obstacle_rect = pygame.Rect(obstacle_x, obstacle_y, obstacle_width, obstacle_height)
            obstacles.append((new_obstacle_rect, random.choice([RED, BLUE, GRAY]))) # Add new obstacle with a random color

        # Obstacle movement and removal
        obstacles_to_remove = []
        for i, (obstacle_rect, color) in enumerate(obstacles):
            obstacle_rect.x -= game_speed # Move obstacle to the left
            if obstacle_rect.right < 0:
                obstacles_to_remove.append(i) # Mark obstacles that are completely off-screen

        # Remove obstacles that are off-screen (iterate in reverse to avoid index issues)
        for i in reversed(obstacles_to_remove):
            obstacles.pop(i)

        # Collision detection: Check if player hits any obstacle
        for obstacle_rect, color in obstacles:
            if player_rect.colliderect(obstacle_rect):
                game_over = True # End the game on collision
                break

        # Score update: Increase score based on time survived
        score += 1 / FPS
        # Gradually increase game speed to add difficulty
        game_speed += SPEED_INCREASE_RATE

    # --- Drawing ---
    screen.fill(WHITE) # Fill background with white

    # Draw ground
    pygame.draw.rect(screen, GREEN, (0, SCREEN_HEIGHT - GROUND_HEIGHT, SCREEN_WIDTH, GROUND_HEIGHT))

    # Draw player
    pygame.draw.rect(screen, BLACK, player_rect)

    # Draw obstacles
    for obstacle_rect, color in obstacles:
        pygame.draw.rect(screen, color, obstacle_rect)

    # Draw score display
    score_text = font.render(f"Score: {int(score)}", True, BLACK)
    screen.blit(score_text, (10, 10))

    # Draw Game Over screen if the game has ended
    if game_over:
        game_over_text = game_over_font.render("GAME OVER", True, RED)
        final_score_text = font.render(f"Final Score: {int(score)}", True, BLACK)
        restart_text = font.render("Press SPACE or UP to Play Again", True, BLACK)

        # Center the "GAME OVER" text
        text_rect = game_over_text.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 - 40))
        # Center the final score text
        score_rect = final_score_text.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2))
        # Center the restart instruction text
        restart_rect = restart_text.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 40))

        screen.blit(game_over_text, text_rect)
        screen.blit(final_score_text, score_rect)
        screen.blit(restart_text, restart_rect)

    # --- Update Display ---
    pygame.display.flip()

    # --- Frame Rate Control ---
    clock.tick(FPS)

# --- These lines are removed for pygbag compatibility ---
# pygame.quit()
# sys.exit()

Overwriting main.py


To resolve the persistent `main.py` path issue with `pygbag`, we will create a dedicated project directory and move `main.py` into it.

In [53]:
# Create a dedicated directory for the game project
!mkdir -p /content/game_project

Now, we will move the `main.py` file into the newly created `/content/game_project` directory. The `%%writefile` command below will automatically overwrite `main.py` in its new location.

In [54]:
%%writefile /content/game_project/main.py
import pygame
import sys
import random

# --- Pygame Initialization ---
pygame.init()

# --- Constants ---
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 400
GROUND_HEIGHT = 50  # Height of the ground from the bottom of the screen
FPS = 60            # Frames per second

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
BLUE = (0, 0, 255)
GREEN = (0, 255, 0)
RED = (255, 0, 0)
GRAY = (150, 150, 150)

# Player properties
PLAYER_WIDTH = 30
PLAYER_HEIGHT = 50
PLAYER_X = 50  # Player's fixed horizontal position on the screen

# Jump properties
JUMP_STRENGTH = -12 # Negative for upward movement
GRAVITY = 0.5       # Gravity pulling the player down

# Obstacle properties
OBSTACLE_WIDTH_MIN = 20
OBSTACLE_WIDTH_MAX = 70
OBSTACLE_HEIGHT_MIN = 30
OBSTACLE_HEIGHT_MAX = 80
# These define the horizontal distance from the left edge of the screen that the
# *left edge* of the last obstacle must cross before a new one can be generated off-screen.
# This indirectly controls the density and spacing of obstacles.
MIN_DISTANCE_BETWEEN_OBSTACLES = 200
MAX_DISTANCE_BETWEEN_OBSTACLES = 400

# Game speed
INITIAL_GAME_SPEED = 5
SPEED_INCREASE_RATE = 0.001 # How much game speed increases per frame

# --- Screen Setup ---
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Pixel Dash")
clock = pygame.time.Clock()

# --- Fonts ---
font = pygame.font.Font(None, 36)         # Default font for score and instructions
game_over_font = pygame.font.Font(None, 64) # Larger font for "GAME OVER"

# --- Game Variables (initial state) ---
game_over = False
score = 0
game_speed = INITIAL_GAME_SPEED

# Player state
# Player's starting y position (on top of the ground)
player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
player_vel_y = 0       # Player's vertical velocity
is_jumping = False     # True if the player is currently in the air
# Pygame Rect for player, used for drawing and collision detection.
# For simplicity, player animation is represented by this rectangle.
player_rect = pygame.Rect(PLAYER_X, player_y, PLAYER_WIDTH, PLAYER_HEIGHT)

# List to store active obstacles. Each obstacle is a tuple: (pygame.Rect, color)
obstacles = []


# --- Helper Function to Reset Game ---
def reset_game():
    """Resets all game variables to their initial state for a new game."""
    global game_over, score, game_speed, player_y, player_vel_y, is_jumping, obstacles
    game_over = False
    score = 0
    game_speed = INITIAL_GAME_SPEED
    player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
    player_vel_y = 0
    is_jumping = False
    obstacles = []
    player_rect.y = player_y # Ensure player_rect is updated for the new game

# --- Main Game Loop ---
running = True
while running:
    # --- Event Handling ---
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE or event.key == pygame.K_UP:
                if game_over:
                    # If game is over, pressing jump restarts the game
                    reset_game()
                elif not is_jumping:
                    # Only allow jump if the player is on the ground
                    player_vel_y = JUMP_STRENGTH
                    is_jumping = True

    # --- Game Logic (only runs if game is not over) ---
    if not game_over:
        # Player update: Apply gravity and update position
        player_vel_y += GRAVITY
        player_y += player_vel_y

        # Prevent player from falling below the ground
        if player_y >= SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT:
            player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
            player_vel_y = 0
            is_jumping = False

        # Update the player's collision rectangle
        player_rect.y = player_y

        # Obstacle generation logic
        # Generate a new obstacle if there are no obstacles, or if the last obstacle
        # has moved far enough to the left to allow for a new one with proper spacing.
        # The random.randint here defines how far to the left of SCREEN_WIDTH the
        # *left edge* of the last obstacle must be before a new one can spawn.
        if not obstacles or obstacles[-1][0].x < SCREEN_WIDTH - random.randint(MIN_DISTANCE_BETWEEN_OBSTACLES, MAX_DISTANCE_BETWEEN_OBSTACLES):
            obstacle_width = random.randint(OBSTACLE_WIDTH_MIN, OBSTACLE_WIDTH_MAX)
            obstacle_height = random.randint(OBSTACLE_HEIGHT_MIN, OBSTACLE_HEIGHT_MAX)
            # Start obstacles slightly off-screen to the right
            obstacle_x = SCREEN_WIDTH + random.randint(0, 100)
            obstacle_y = SCREEN_HEIGHT - GROUND_HEIGHT - obstacle_height
            new_obstacle_rect = pygame.Rect(obstacle_x, obstacle_y, obstacle_width, obstacle_height)
            obstacles.append((new_obstacle_rect, random.choice([RED, BLUE, GRAY]))) # Add new obstacle with a random color

        # Obstacle movement and removal
        obstacles_to_remove = []
        for i, (obstacle_rect, color) in enumerate(obstacles):
            obstacle_rect.x -= game_speed # Move obstacle to the left
            if obstacle_rect.right < 0:
                obstacles_to_remove.append(i) # Mark obstacles that are completely off-screen

        # Remove obstacles that are off-screen (iterate in reverse to avoid index issues)
        for i in reversed(obstacles_to_remove):
            obstacles.pop(i)

        # Collision detection: Check if player hits any obstacle
        for obstacle_rect, color in obstacles:
            if player_rect.colliderect(obstacle_rect):
                game_over = True # End the game on collision
                break

        # Score update: Increase score based on time survived
        score += 1 / FPS
        # Gradually increase game speed to add difficulty
        game_speed += SPEED_INCREASE_RATE

    # --- Drawing ---
    screen.fill(WHITE) # Fill background with white

    # Draw ground
    pygame.draw.rect(screen, GREEN, (0, SCREEN_HEIGHT - GROUND_HEIGHT, SCREEN_WIDTH, GROUND_HEIGHT))

    # Draw player
    pygame.draw.rect(screen, BLACK, player_rect)

    # Draw obstacles
    for obstacle_rect, color in obstacles:
        pygame.draw.rect(screen, color, obstacle_rect)

    # Draw score display
    score_text = font.render(f"Score: {int(score)}", True, BLACK)
    screen.blit(score_text, (10, 10))

    # Draw Game Over screen if the game has ended
    if game_over:
        game_over_text = game_over_font.render("GAME OVER", True, RED)
        final_score_text = font.render(f"Final Score: {int(score)}", True, BLACK)
        restart_text = font.render("Press SPACE or UP to Play Again", True, BLACK)

        # Center the "GAME OVER" text
        text_rect = game_over_text.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 - 40))
        # Center the final score text
        score_rect = final_score_text.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2))
        # Center the restart instruction text
        restart_rect = restart_text.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 40))

        screen.blit(game_over_text, text_rect)
        screen.blit(final_score_text, score_rect)
        screen.blit(restart_text, restart_rect)

    # --- Update Display ---
    pygame.display.flip()

    # --- Frame Rate Control ---
    clock.tick(FPS)

# --- These lines are removed for pygbag compatibility ---
# pygame.quit()
# sys.exit()

Overwriting /content/game_project/main.py


Now that `main.py` is in `/content/game_project/`, we need to update the `pygbag` command to point to this new project directory. I will modify cell `ey6GoyYxJWIq` to reflect this change.

In [55]:
!pip install pygbag pyngrok -q

# **How to get pyngrok API Key?**
- Go to https://dashboard.ngrok.com/get-started/your-authtoken
- Copy your authtoken
- Run this cell with YOUR token:

In [56]:
!ngrok authtoken '3Etc9zE2pWRV673jQLOBkyFpKCv_7XoQmCvvnmi569GEXPJ4d'

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [60]:
import subprocess
import time
from pyngrok import ngrok
import os
import psutil

# Define the path for the HTTP server log
HTTP_SERVER_LOG_FILE = '/tmp/http_server_log.txt'

# Step 1: Build the game
print("🔨 Building game...")
build_result = subprocess.run(
    ['python', '-m', 'pygbag', '--build', '--version', '0.9', '--PYBUILD', '3.12', '--cdn', 'https://pygame-web.github.io/archives/0.9/', '.'],
    capture_output=True, text=True,
    cwd='/content/game_project/' # Set current working directory for pygbag to its project root
)
print(build_result.stdout)

if build_result.returncode != 0:
    print("❌ Build failed:")
    print(build_result.stderr)
else:
    print("✅ Build successful!")

    # Step 2: Stop any existing server on port 8000
    print("\nStopping any existing server on port 8000...")
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        try:
            # Check if the process is a python http.server and listening on port 8000
            cmdline = ' '.join(proc.info['cmdline'])
            if 'python' in proc.info['name'] and 'http.server' in cmdline and '8000' in cmdline:
                print(f"  Killing process {proc.info['pid']} ({cmdline})")
                proc.terminate()
                time.sleep(1) # Give it a moment to terminate
                if proc.is_running():
                    proc.kill()
                    print(f"  Killed process {proc.info['pid']}")
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            pass
    print("Done stopping servers.")

    # Step 2: Start HTTP server and redirect output to log file using Popen
    print(f"\n🚀 Starting HTTP server (port 8000) and logging to {HTTP_SERVER_LOG_FILE}...")

    # Ensure the log file is empty before starting
    with open(HTTP_SERVER_LOG_FILE, 'w') as f:
        f.write('')

    # Start the server using Popen, redirecting stdout/stderr to the log file
    server_log_file_handle = open(HTTP_SERVER_LOG_FILE, 'a') # Open in append mode
    server = subprocess.Popen(
        ['python', '-m', 'http.server', '8000'],
        stdout=server_log_file_handle,
        stderr=server_log_file_handle,
        cwd='/content/game_project/build/web' # Server will serve from this directory
    )

    # Give the server a moment to start and log any initial messages
    time.sleep(5)

    if server.poll() is not None: # Check if the process has terminated
        print(f"\n❌ HTTP server process terminated unexpectedly with exit code {server.returncode}")
        server_log_file_handle.close()
        with open(HTTP_SERVER_LOG_FILE, 'r') as f:
            print("Server log:")
            print(f.read())
        # Optionally exit or raise an error here if server MUST be running
    else:
        print("✅ HTTP server appears to be running.")

        # Step 3: Create ngrok tunnel
        try:
            print("🌐 Disconnecting existing ngrok tunnels (if any)...")
            ngrok.kill() # Disconnect all tunnels from previous runs

            print("🌐 Creating public URL...")
            public_url = ngrok.connect(8000)

            print("\n" + "="*60)
            print("🎮 YOUR GAME IS READY!")
            print("="*60)
            print(f"\n🔗 Click here to play: {public_url}")
            print("\n📝 How to play:")
            print("   • Press SPACE or Click to jump")
            print("="*60)

        except Exception as e:
            print(f"\n❌ Error creating tunnel: {e}")
            print("\n💡 Make sure you ran Cell 3 with a valid ngrok token and consider your ngrok plan's tunnel limits.")
        finally:
            # Close the server log file handle
            server_log_file_handle.close()
            # Do not terminate server here, it's needed for ngrok to work


🔨 Building game...
 *pygbag 0.9.3*

Serving python files from [/content/game_project/build/web]

with no security/performance in mind, i'm just a test tool : don't rely on me


SUMMARY
________________________

# the app folder
app_folder=/content/game_project

# artefacts directory
build_dir=/content/game_project/build/web

# cache directory
cache=/content/game_project/build/web-cache

# the window title and icon name
app_name=game_project

# package name, better make it unique
package=web.pygame.game_project-1781331755

# icons:  96x96 for desktop, 16x16 for web
icon=favicon.png

# js/wasm provider
cdn=https://pygame-web.github.io/archives/0.9/

now packing application ....


Ignored dirs: []
Ignored files: []
Now in /
     /main.py
optimizing /content/game_project
	/content/game_project : main.py
	/content/game_project : main.py
packing 1 files complete

    building from local cached template https://pygame-web.github.io/archives/0.9/default.tmpl
    cached at /content/game_project

In [61]:
print("HTTP Server Log:")
!cat /tmp/http_server_log.txt

HTTP Server Log:
